In [1]:
print("hi")

hi


In [1]:
import os
import json
import re
import pdfplumber
from pypdf import PdfReader, PdfWriter
from PIL import Image
# from pipe_fn import pipe
from transformers import pipeline
from output_utils import save_split_output

from confidence_utils import (
    calculate_eob_confidence,
    _unwrap_vlm_output,calculate_model_confidence
)
# =========================================================
# LOAD MODEL
# =========================================================

pipe = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device_map="auto"
)


COLUMN_HEADERS = [
    "date_of_service",
    "procedure_code",
    "total_charge",
    "plan_allowed_amount",
    "deductible_amount",
    "ineligible_amount",
    "previous_pay",
    "refund_amount",
    "claim_paid",
    "patient_responsibility"
]



def rotate_pdf_except_first(pdf_path, output_dir, angle=90):
    """
    Rotate all pages except the first page and save a temporary rotated PDF.
    """
    os.makedirs(output_dir, exist_ok=True)

    pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
    rotated_pdf_path = os.path.join(output_dir, f"{pdf_name}_rotated.pdf")

    reader = PdfReader(pdf_path)
    writer = PdfWriter()

    for page_num, page in enumerate(reader.pages):
        print(f"Processing Page {page_num + 1}")

        if page_num > 0:
            page.rotate(angle)
            print(f"   ↳ Rotated {angle}°")
        else:
            print("   ↳ Kept original")

        writer.add_page(page)

    with open(rotated_pdf_path, "wb") as f:
        writer.write(f)

    print(f"\nRotated PDF saved: {rotated_pdf_path}")

    return rotated_pdf_path


def enforce_schema(parsed_output):

    allowed_columns = COLUMN_HEADERS  # ✅ ordered list

    for table in parsed_output.get("tables", []):

        # =========================
        # 🔹 CLEAN ROWS (STRICT + ORDERED)
        # =========================
        cleaned_rows = []

        for row in table.get("rows", []):

            cleaned_row = {}

            for col in allowed_columns:
                cleaned_row[col] = row.get(col, "")

            procedure_code = cleaned_row.get(
                "procedure_code",
                ""
            ).strip()

            if not re.match(r"^D\d{4}$", procedure_code):
                continue

            cleaned_rows.append(cleaned_row)
        table["rows"] = cleaned_rows

        # =========================
        # 🔹 CLEAN COLUMN TOTALS (STRICT + ORDERED)
        # =========================
        totals = table.get("column_totals", {})
        ordered_totals = {}

        for col in allowed_columns:
            if col not in ["date_of_service", "procedure_code"]:
                ordered_totals[col] = totals.get(col, "")

        table["column_totals"] = ordered_totals

    return parsed_output



PROMPT = """
You are extracting structured financial data from a cropped Dental EOB table image.

STRICT EXTRACTION RULES

=========================================================
1. SERVICE ROW EXTRACTION
=========================================================

Extract ONLY the visible service rows.

DO NOT:
- hallucinate rows
- duplicate rows
- merge rows
- restart extraction
- infer hidden rows

Each extracted row MUST correspond to exactly one visible service row.

Stop extracting service rows immediately when any of these is reached:

- CLAIM TOTALS
- Column Totals

Never include the totals row as a service row.

=========================================================
2. PROCEDURE CODE
=========================================================

Extract only valid ADA procedure codes.

Format:

D followed by exactly four digits.

Examples:

D7140
D4341
D1110

If no valid code exists return "".

=========================================================
3. DATE OF SERVICE
=========================================================

Extract only the date.

Example:

08/02/2022

Ignore all surrounding text.

=========================================================
4. MONEY COLUMNS
=========================================================

Extract ONLY the numeric amount.

Columns:

- total_charge
- plan_allowed_amount
- deductible_amount
- ineligible_amount
- previous_pay
- refund_amount
- claim_paid
- patient_responsibility

Rules

- Remove $
- Remove commas
- Preserve decimal places
- Return values like

155.00
62.05
0.00

If empty return "".

Never guess missing values.

=========================================================
5. CLAIM TOTALS
=========================================================

Extract totals ONLY from the row labeled

CLAIM TOTALS

Do NOT calculate totals.

Map totals only to their matching columns.

Example

"column_totals": {
    "total_charge":"",
    "plan_allowed_amount":"",
    "deductible_amount":"",
    "ineligible_amount":"",
    "previous_pay":"",
    "refund_amount":"",
    "claim_paid":"",
    "patient_responsibility":""
}

=========================================================
6. COLUMN ALIGNMENT
=========================================================

Never shift values between columns.

Each value must come ONLY from its own column.

Example

Plan Allowed Amount must never be copied into Deductible.

Claim Paid must never be copied into Patient Responsibility.

=========================================================
7. DATA INTEGRITY
=========================================================

Do NOT

- infer values
- modify values
- perform calculations
- normalize numbers
- invent remark codes

Extract exactly what is visible.

=========================================================
8. OUTPUT FORMAT
=========================================================
Return ONLY valid JSON.

{
    "rows": [
        {
            "date_of_service": {
                "value": "",
                "confidence": 0.0
            },

            "procedure_code": {
                "value": "",
                "confidence": 0.0
            },

            "total_charge": {
                "value": "",
                "confidence": 0.0
            },

            "plan_allowed_amount": {
                "value": "",
                "confidence": 0.0
            },

            "deductible_amount": {
                "value": "",
                "confidence": 0.0
            },

            "ineligible_amount": {
                "value": "",
                "confidence": 0.0
            },

            "previous_pay": {
                "value": "",
                "confidence": 0.0
            },

            "refund_amount": {
                "value": "",
                "confidence": 0.0
            },

            "claim_paid": {
                "value": "",
                "confidence": 0.0
            },

            "patient_responsibility": {
                "value": "",
                "confidence": 0.0
            }
        }
    ],

    "column_totals": {
        "total_charge": {
            "value": "",
            "confidence": 0.0
        },

        "plan_allowed_amount": {
            "value": "",
            "confidence": 0.0
        },

        "deductible_amount": {
            "value": "",
            "confidence": 0.0
        },

        "ineligible_amount": {
            "value": "",
            "confidence": 0.0
        },

        "previous_pay": {
            "value": "",
            "confidence": 0.0
        },

        "refund_amount": {
            "value": "",
            "confidence": 0.0
        },

        "claim_paid": {
            "value": "",
            "confidence": 0.0
        },

        "patient_responsibility": {
            "value": "",
            "confidence": 0.0
        }
    }
}

 For every extracted field, return:
   - value
   - confidence
 
VALUE + CONFIDENCE RULES:
 
For every field return:
{
  "value": "",
  "confidence": ""
}
 
VALUE:
- "value" = the exact text/value visibly present in the specified location.
- Read ONLY from the exact cell/row/column requested.
- Copy exactly as printed; preserve "$" and formatting when visible.
- Never guess, infer, calculate, copy, shift, or use values from another row,
  column, table section, or Totals row.
- If the exact location is blank, missing, or has no clearly readable value:
  value = ""
 
CONFIDENCE:
- "confidence" = confidence that the extracted value is actually present
  in that exact location.
- Use a number from 0.0 to 1.0 based ONLY on visual evidence.
- 1.0 = clearly visible and certain.
- 0.8–0.99 = clearly visible with minor uncertainty.
- 0.5–0.79 = visible but difficult/ambiguous.
- 0.1–0.49 = very unclear.
- 0.0 = blank, missing, or no reliable visual evidence.
 
IMPORTANT:
Confidence is NOT confidence that the value is mathematically correct
or logically expected. It is ONLY confidence that the value shown in
"value" is what is visibly printed in the exact requested location.
 
If value = "":
confidence MUST = 0.0.

=========================================================
11. PRIORITY
=========================================================

Accuracy > completeness.

If a value cannot be read confidently,
return "" instead of guessing.
"""


def extract_table_from_image(image_path, expected_rows=None):

    image = Image.open(image_path).convert("RGB")

    row_instruction = ""

    if expected_rows is not None:
        row_instruction = f"""

=========================================================
IMPORTANT PHYSICAL ROW COUNT
=========================================================

An independent PDF analysis detected EXACTLY {expected_rows}
physical service row(s) in this image.

You MUST return exactly {expected_rows} objects inside "rows".

IMPORTANT:
- Count PHYSICAL service rows.
- Do NOT count the CLAIM TOTALS row.
- Do NOT count the header row.
- Do NOT create extra rows.
- Do NOT duplicate a physical row.
- Do NOT merge physical rows.
- Two physically separate rows MAY contain identical values.
- If two physical rows have identical values, KEEP BOTH.
- Never remove a row just because its values are identical to another row.

The number of objects in "rows" MUST be exactly {expected_rows}.
"""

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image
                },
                {
                    "type": "text",
                    "text": PROMPT + row_instruction
                }
            ]
        }
    ]

    output = pipe(
        text=messages,
        max_new_tokens=5000,
        temperature=0.0,
        repetition_penalty=1.2
    )

    generated_text = output[0]["generated_text"]

    if isinstance(generated_text, list):

        last_message = generated_text[-1]

        if isinstance(last_message, dict):
            generated_text = last_message.get("content", "")
        else:
            generated_text = last_message

    if isinstance(generated_text, list):

        contents = []

        for item in generated_text:
            if isinstance(item, dict):
                text = item.get("text") or item.get("content") or ""

                if isinstance(text, str):
                    contents.append(text)

            elif isinstance(item, str):
                contents.append(item)

        generated_text = "".join(contents)

    elif isinstance(generated_text, dict):

        generated_text = (
            generated_text.get("text")
            or generated_text.get("content")
            or ""
        )

    if not isinstance(generated_text, str):
        generated_text = str(generated_text)

    generated_text = generated_text.strip()

    generated_text = generated_text.replace(
        "```json",
        ""
    ).replace(
        "```",
        ""
    ).strip()

    return generated_text



def parse_amount(val):
    if val in ["", None]:
        return 0.0
    return float(str(val).replace("$", "").replace(",", "").strip())


def count_service_rows(page, region_top, region_bottom):

    words = page.extract_words()
    row_positions = []

    for w in words:
        text = w["text"].strip()

        match = re.search(r"\bD\d{4}\b", text)

        if match:
            y = float(w["top"])

            if region_top <= y <= region_bottom:
                row_positions.append(y)

    row_positions.sort()

    grouped_rows = []
    threshold = 3

    for y in row_positions:
        if not grouped_rows:
            grouped_rows.append(y)
        else:
            if abs(y - grouped_rows[-1]) > threshold:
                grouped_rows.append(y)

    return len(grouped_rows)


def extract_patient_name(page):
    text = page.extract_text()

    if not text:
        return ""

    match = re.search(
        r"Patient:\s*(.+?)(?=\s{2,}|Member ID|\n|$)",
        text,
        re.IGNORECASE
    )

    if match:
        name = match.group(1).strip()
        return name

    return ""

def check_claim_denied(pdf_path):

    denial_keywords = [
        "denied",
        "denial"
    ]

    with pdfplumber.open(pdf_path) as pdf:

        for page_num, page in enumerate(pdf.pages, start=1):

            full_text = page.extract_text()

            if not full_text:
                continue

            searchable_text = full_text.lower()

            for keyword in denial_keywords:

                if keyword in searchable_text:

                    print(
                        f"❌ Claim denied keyword found: "
                        f"'{keyword}' on page {page_num}"
                    )

                    return "denied"

    return "not denied"


def run_pipeline(
    pdf_path,
    output_dir="EOB_OUTPUT/careington_results",
    rotate_angle=90,
    start_anchor="Claim #",
    end_anchor="CLAIM TOTALS", company_name = "Careington Benefits"
):
    """
    Full pipeline:
        1. Rotate all pages except first.
        2. Open rotated PDF.
        3. Crop each Claim block.
        4. Save images.
        5. Run VLM extraction on each crop.
        6. Validate + build final JSON.

    Output structure:

    careington_results/
        sample_pdf/
            sample_pdf_rotated.pdf
            page_2_block_1.png
            page_2_block_2.png
            ...
            results.json
    """

    pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
    pdf_full_name = os.path.basename(pdf_path)

    working_dir = os.path.join(output_dir, pdf_name)

    os.makedirs(working_dir, exist_ok=True)

    # -------------------------------------------------------
    # Step 1 : Rotate PDF
    # -------------------------------------------------------
    rotated_pdf_path = rotate_pdf_except_first(
        pdf_path=pdf_path,
        output_dir=working_dir,
        angle=rotate_angle
    )
    claim_status = check_claim_denied(rotated_pdf_path)
    print(f"Claim Status: {claim_status}")

    results = []
    all_patients = []

    # -------------------------------------------------------
    # Step 2 : Crop Rotated PDF
    # -------------------------------------------------------
    with pdfplumber.open(rotated_pdf_path) as pdf:

        for page_idx, page in enumerate(pdf.pages):

            print(f"\nScanning Page {page_idx + 1}")

            start_matches = page.search(re.escape(start_anchor))
            end_matches = page.search(re.escape(end_anchor))

            pt_positions = [m["top"] for m in start_matches]
            end_positions = [m["bottom"] for m in end_matches]

            # print(f"Start anchors found : {len(pt_positions)}")
            # print(f"End anchors found   : {len(end_positions)}")

            if not pt_positions or not end_positions:
                print("Skipping page - anchors not found.")
                continue

            end_idx = 0

            for block_idx, start_top in enumerate(pt_positions, start=1):

                while (
                    end_idx < len(end_positions)
                    and end_positions[end_idx] <= start_top
                ):
                    end_idx += 1

                if end_idx >= len(end_positions):
                    break

                crop_top = max(start_top - 8, 0)
                crop_bottom = end_positions[end_idx] + 10

                bbox = (
                    0,
                    crop_top,
                    page.width,
                    crop_bottom
                )

                cropped = page.crop(bbox)

                output_path = os.path.join(
                    working_dir,
                    f"page_{page_idx + 1}_block_{block_idx}.png"
                )

                cropped.to_image(resolution=300).save(output_path)

                print(f"Saved : {output_path}")

                results.append({
                    "page": page_idx + 1,
                    "block": block_idx,
                    "file": output_path
                })

                end_idx += 1

                # =====================================
                # STEP 3 : VLM EXTRACTION + VALIDATION 
                # =====================================
                try:

                    # =====================================
                    # COUNT PHYSICAL SERVICE ROWS FIRST
                    # =====================================
                    expected_rows = count_service_rows(
                        page,
                        crop_top,
                        crop_bottom
                    )

                    print(
                        f"📌 Expected physical service rows: {expected_rows}"
                    )


                    # =====================================
                    # VLM EXTRACTION
                    # =====================================
                    llm_output = extract_table_from_image(
                        output_path,
                        expected_rows=expected_rows
                    )

                    # print("\n================ RAW MODEL OUTPUT ================\n")
                    # print(llm_output)
                    # print("\n==================================================\n")

                    parsed_output = json.loads(llm_output)

                    model_confidence = calculate_model_confidence(parsed_output)

                    parsed_output = _unwrap_vlm_output(parsed_output)

                    parsed_output = {
                        "tables": [parsed_output]
                    }

                    parsed_output = enforce_schema(parsed_output)

                    # =====================================
                    # ENFORCE EXPECTED ROW COUNT
                    # =====================================
                    for table in parsed_output.get("tables", []):

                        rows = table.get("rows", [])

                        if expected_rows is not None and len(rows) > expected_rows:

                            print(
                                f"⚠️ VLM returned {len(rows)} rows, "
                                f"but PDF detected {expected_rows} rows."
                            )

                            # IMPORTANT:
                            # Do NOT deduplicate rows.
                            # Identical physical rows are valid.
                            rows = rows[:expected_rows]

                            table["rows"] = rows

                    patient_name = extract_patient_name(page)
                    print(f" Patient Name: {patient_name}")

                    for table in parsed_output.get("tables", []):

                        new_table = {
                            "EOB_ID": pdf_name,
                            "patient_name": patient_name,
                            "rows": table.get("rows", []),
                            "column_totals": table.get("column_totals", {}),
                            "_model_confidence": model_confidence, 
                        }

                        table.clear()
                        table.update(new_table)

                    for t_idx, table in enumerate(parsed_output.get("tables", []), start=1):

                        row_count_ok = validate_service_row_count(
                            page,
                            crop_top,
                            crop_bottom,
                            table,
                            t_idx
                        )
                    # expected_rows = count_service_rows(page, crop_top, crop_bottom)

                    for t_idx, table in enumerate(parsed_output.get("tables", []), start=1):

                        
                        is_valid, log, errors, total_fields  = validate_eob_table(table, t_idx)
                        if not row_count_ok:
                            is_valid = False,
                            errors = errors + [{"type": "row_count_mismatch"}]


                    for table in parsed_output.get("tables", []):

                        structured_table = {
                            "EOB_ID": pdf_name,
                            "patient_name": table.get("patient_name", ""),
                            "rows": table.get("rows", []),
                            "totals": table.get("column_totals", {}),
                            "validation": {"status": is_valid, "errors": errors}, 
                            "_expected_rows": expected_rows,                          # ADD
                            "_total_fields": total_fields,                             # ADD
                            "_model_confidence": table.get("_model_confidence", 0.0),  # ADD
                        }

                        print(f"✅ Completed Table {block_idx} on Page {page_idx + 1}")

                    date_of_service = ""

                    if structured_table.get("rows"):
                        date_of_service = structured_table["rows"][0].get("date_of_service", "")

                    for row in structured_table.get("rows", []):

                        for col in ["service_description", "date_of_service"]:
                            row.pop(col, None)

                    patient_data = {
                        "patient_name": structured_table.get("patient_name", ""),
                        "date_of_service": date_of_service,
                        "services": structured_table.get("rows", []),
                        "totals": structured_table.get("totals", {}),
                        "validation": structured_table.get("validation"),
                        "_expected_rows": structured_table.get("_expected_rows", 0),               # ADD
                        "_total_fields": structured_table.get("_total_fields", 0),                  # ADD
                        "_model_confidence": structured_table.get("_model_confidence", 0.0), 
                    }

                    all_patients.append(patient_data)

                except Exception as e:
                    print(f"Error: {e}")

    print("\n===================================")
    print(f"Total Crops Created : {len(results)}")
    print("===================================")

    confidence_score = calculate_eob_confidence(all_patients)   # ADD

    for patient in all_patients:                                 # ADD cleanup
        patient.pop("_expected_rows", None)
        patient.pop("_total_fields", None)
        patient.pop("_model_confidence", None)

    # -------------------------------------------------------
    # Step 4 : Save final JSON
    # -------------------------------------------------------
    final_output = [
        {
            "eob_id": pdf_name,
            "file_name":pdf_full_name,
            "claim_status": claim_status,
            "payor": "CAREINGTON BENEFIT SOLUTIONS",
            "confidence_score": confidence_score, 
            "patients": all_patients
        }
    ]

    success_path, failed_path = save_split_output(
                            final_output,
                            company_name=company_name,
                            pdf_name=pdf_name,
                            pdf_path=pdf_path,
                            cropped_dir=working_dir,
                        )
                    
    print(f"\n📁 Cropped images : {working_dir}")
    print(f"✅ Success json   : {success_path}")
    print(f"⚠  Failed json    : {failed_path}")
    return final_output

    # return results, final_output


def validate_eob_table(table: dict, table_index: int):

    rows = table.get("rows", [])
    totals = table.get("column_totals", {})

    if not rows:
        return False, "", [], 0

    computed_totals = {
        "total_charge": round(sum(parse_amount(r.get("total_charge", "")) for r in rows), 2),
        "plan_allowed_amount": round(sum(parse_amount(r.get("plan_allowed_amount", "")) for r in rows), 2),
        "deductible_amount": round(sum(parse_amount(r.get("deductible_amount", "")) for r in rows), 2),
        # "copay_amount": round(sum(parse_amount(r.get("copay_amount", "")) for r in rows), 2),
        "ineligible_amount": round(sum(parse_amount(r.get("ineligible_amount", "")) for r in rows), 2),
        "previous_pay": round(sum(parse_amount(r.get("previous_pay", "")) for r in rows), 2),
        "refund_amount": round(sum(parse_amount(r.get("refund_amount", "")) for r in rows), 2),
        "claim_paid": round(sum(parse_amount(r.get("claim_paid", "")) for r in rows), 2),
        "patient_responsibility": round(sum(parse_amount(r.get("patient_responsibility", "")) for r in rows), 2)
    }

    total_fields = len(computed_totals) 

    result_validation = ""
    errors = []
    has_error = False

    print(f"\n🔍 Validation for [Table {table_index}]")
    print("-" * 75)

    for field, computed_value in computed_totals.items():

        extracted_value = round(parse_amount(totals.get(field, "")), 2)

        if computed_value == extracted_value:
            icon = "✅"
            status = "match"
        else:
            icon = "❌"
            status = "MISMATCH"
            has_error = True

            errors.append({
                "field": field,
                "computed": computed_value,
                "extracted": extracted_value
            })

        line = f"{icon} {field:25s} computed={computed_value:<10} | extracted={extracted_value:<10} {status}"
        print(line)
        result_validation += "\n" + line

    if has_error:
        print(f"❌ [Table {table_index}] Validation FAILED\n")
        return False, result_validation, errors, total_fields    
    else:
        print(f"✅ [Table {table_index}] Validation PASSED\n")
        return True, result_validation, [], total_fields    
    

def validate_service_row_count(page, start_y, end_y, table, table_index):

    detected_count = count_service_rows(page, start_y, end_y)

    rows = table.get("rows", [])
    extracted_count = len([
        r for r in rows
        if r.get("procedure_code") not in ["", None]
    ])

    print(f"\n📊 Row Count Validation [Table {table_index}]")
    print("-" * 70)

    if detected_count == extracted_count:
        icon = "✅"
        status = "match"
    else:
        icon = "❌"
        status = "MISMATCH"

    print(f"{icon} row_count detected={detected_count:<5} | extracted={extracted_count:<5} {status}")
    print("-" * 70)


    return detected_count == extracted_count

W0901 18:37:16.223000 3444653 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0901 18:37:16.238000 3444653 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

# Test 1

In [2]:
import os

folder_path = r"/home/cipl/users/OCR_Project/Careington/pdfs"

for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(folder_path, filename)

        print(f"\n{'='*80}")
        print(f"Processing: {filename}")
        print(f"{'='*80}")

        try:
            run_pipeline(pdf_path)
            print(f"✅ Completed: {filename}")

        except Exception as e:
            print(f"❌ Failed: {filename}")
            print(f"Error: {e}")


Processing: Pmt_EOP_271302227.pdf
Processing Page 1
   ↳ Kept original
Processing Page 2
   ↳ Rotated 90°
Processing Page 3
   ↳ Rotated 90°

Rotated PDF saved: EOB_OUTPUT/careington_results/Pmt_EOP_271302227/Pmt_EOP_271302227_rotated.pdf
Claim Status: not denied

Scanning Page 1
Skipping page - anchors not found.

Scanning Page 2
Skipping page - anchors not found.

Scanning Page 3


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `repetition_penalty` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved : EOB_OUTPUT/careington_results/Pmt_EOP_271302227/page_3_block_1.png
 Patient Name: CHARLOTTE DEBOER

📊 Row Count Validation [Table 1]
----------------------------------------------------------------------
❌ row_count detected=4     | extracted=5     MISMATCH
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
❌ total_charge              computed=775.0      | extracted=620.0      MISMATCH
❌ plan_allowed_amount       computed=715.0      | extracted=572.0      MISMATCH
✅ deductible_amount         computed=0.0        | extracted=0.0        match
✅ copay_amount              computed=0.0        | extracted=0.0        match
❌ ineligible_amount         computed=310.25     | extracted=248.2      MISMATCH
✅ previous_pay              computed=0.0        | extracted=0.0        match
✅ refund_amount             computed=0.0        | extracted=0.0        match
❌ claim_pai

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved : EOB_OUTPUT/careington_results/Pmt_EOP_314231639/page_3_block_1.png
 Patient Name: GARY CATTEL

📊 Row Count Validation [Table 1]
----------------------------------------------------------------------
✅ row_count detected=2     | extracted=2     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ total_charge              computed=145.0      | extracted=145.0      match
✅ plan_allowed_amount       computed=145.0      | extracted=145.0      match
✅ deductible_amount         computed=0.0        | extracted=0.0        match
✅ copay_amount              computed=0.0        | extracted=0.0        match
✅ ineligible_amount         computed=0.0        | extracted=0.0        match
✅ previous_pay              computed=0.0        | extracted=0.0        match
✅ refund_amount             computed=0.0        | extracted=0.0        match
✅ claim_paid                

...

Saved : EOB_OUTPUT/careington_results/Pmt_EOP_541709991/page_3_block_1.png
 Patient Name: ANGELINA SANTOS

📊 Row Count Validation [Table 1]
----------------------------------------------------------------------
✅ row_count detected=3     | extracted=3     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ total_charge              computed=158.13     | extracted=158.13     match
✅ plan_allowed_amount       computed=102.0      | extracted=102.0      match
✅ deductible_amount         computed=0.0        | extracted=0.0        match
✅ copay_amount              computed=56.13      | extracted=56.13      match
✅ ineligible_amount         computed=56.13      | extracted=56.13      match
✅ previous_pay              computed=0.0        | extracted=0.0        match
✅ refund_amount             computed=0.0        | extracted=0.0        match
✅ claim_paid            

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved : EOB_OUTPUT/careington_results/Pmt_EOP_538836358/page_3_block_1.png
 Patient Name: LYNN MYRICK

📊 Row Count Validation [Table 1]
----------------------------------------------------------------------
✅ row_count detected=5     | extracted=5     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ total_charge              computed=210.0      | extracted=210.0      match
✅ plan_allowed_amount       computed=210.0      | extracted=210.0      match
✅ deductible_amount         computed=41.0       | extracted=41.0       match
✅ copay_amount              computed=0.0        | extracted=0.0        match
✅ ineligible_amount         computed=41.0       | extracted=41.0       match
✅ previous_pay              computed=0.0        | extracted=0.0        match
✅ refund_amount             computed=0.0        | extracted=0.0        match
✅ claim_paid                

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved : EOB_OUTPUT/careington_results/Pmt_EOP_453571987/page_3_block_1.png
 Patient Name: LYNN MYRICK

📊 Row Count Validation [Table 1]
----------------------------------------------------------------------
❌ row_count detected=3     | extracted=4     MISMATCH
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
❌ total_charge              computed=305.0      | extracted=211.0      MISMATCH
❌ plan_allowed_amount       computed=305.0      | extracted=211.0      MISMATCH
❌ deductible_amount         computed=188.0      | extracted=94.0       MISMATCH
✅ copay_amount              computed=0.0        | extracted=0.0        match
❌ ineligible_amount         computed=188.0      | extracted=94.0       MISMATCH
✅ previous_pay              computed=0.0        | extracted=0.0        match
✅ refund_amount             computed=0.0        | extracted=0.0        match
✅ claim_paid 

# Test 2

In [2]:
import os

folder_path = r"/home/cipl/users/OCR_Project/Careington/pdfs"

for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(folder_path, filename)

        print(f"\n{'='*80}")
        print(f"Processing: {filename}")
        print(f"{'='*80}")

        try:
            run_pipeline(pdf_path)
            print(f"✅ Completed: {filename}")

        except Exception as e:
            print(f"❌ Failed: {filename}")
            print(f"Error: {e}")


Processing: Pmt_EOP_271302227.pdf
Processing Page 1
   ↳ Kept original
Processing Page 2
   ↳ Rotated 90°
Processing Page 3
   ↳ Rotated 90°

Rotated PDF saved: EOB_OUTPUT/careington_results/Pmt_EOP_271302227/Pmt_EOP_271302227_rotated.pdf
Claim Status: not denied

Scanning Page 1
Skipping page - anchors not found.

Scanning Page 2
Skipping page - anchors not found.

Scanning Page 3


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `repetition_penalty` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved : EOB_OUTPUT/careington_results/Pmt_EOP_271302227/page_3_block_1.png
📌 Expected physical service rows: 4
 Patient Name: CHARLOTTE DEBOER

📊 Row Count Validation [Table 1]
----------------------------------------------------------------------
✅ row_count detected=4     | extracted=4     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ total_charge              computed=620.0      | extracted=620.0      match
✅ plan_allowed_amount       computed=572.0      | extracted=572.0      match
✅ deductible_amount         computed=0.0        | extracted=0.0        match
✅ copay_amount              computed=0.0        | extracted=0.0        match
✅ ineligible_amount         computed=248.2      | extracted=248.2      match
✅ previous_pay              computed=0.0        | extracted=0.0        match
✅ refund_amount             computed=0.0        | extracted=0.0 

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved : EOB_OUTPUT/careington_results/Pmt_EOP_314231639/page_3_block_1.png
📌 Expected physical service rows: 2
 Patient Name: GARY CATTEL

📊 Row Count Validation [Table 1]
----------------------------------------------------------------------
✅ row_count detected=2     | extracted=2     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ total_charge              computed=145.0      | extracted=145.0      match
✅ plan_allowed_amount       computed=145.0      | extracted=145.0      match
✅ deductible_amount         computed=0.0        | extracted=0.0        match
✅ copay_amount              computed=0.0        | extracted=0.0        match
✅ ineligible_amount         computed=0.0        | extracted=0.0        match
✅ previous_pay              computed=0.0        | extracted=0.0        match
✅ refund_amount             computed=0.0        | extracted=0.0      

...

Saved : EOB_OUTPUT/careington_results/Pmt_EOP_541709991/page_3_block_1.png
📌 Expected physical service rows: 3
 Patient Name: ANGELINA SANTOS

📊 Row Count Validation [Table 1]
----------------------------------------------------------------------
✅ row_count detected=3     | extracted=3     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ total_charge              computed=158.13     | extracted=158.13     match
✅ plan_allowed_amount       computed=102.0      | extracted=102.0      match
✅ deductible_amount         computed=0.0        | extracted=0.0        match
✅ copay_amount              computed=56.13      | extracted=56.13      match
✅ ineligible_amount         computed=56.13      | extracted=56.13      match
✅ previous_pay              computed=0.0        | extracted=0.0        match
✅ refund_amount             computed=0.0        | extracted=0.0  

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved : EOB_OUTPUT/careington_results/Pmt_EOP_538836358/page_3_block_1.png
📌 Expected physical service rows: 5
 Patient Name: LYNN MYRICK

📊 Row Count Validation [Table 1]
----------------------------------------------------------------------
✅ row_count detected=5     | extracted=5     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ total_charge              computed=210.0      | extracted=210.0      match
✅ plan_allowed_amount       computed=210.0      | extracted=210.0      match
✅ deductible_amount         computed=41.0       | extracted=41.0       match
✅ copay_amount              computed=0.0        | extracted=0.0        match
✅ ineligible_amount         computed=41.0       | extracted=41.0       match
✅ previous_pay              computed=0.0        | extracted=0.0        match
✅ refund_amount             computed=0.0        | extracted=0.0      

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved : EOB_OUTPUT/careington_results/Pmt_EOP_453571987/page_3_block_1.png
📌 Expected physical service rows: 3
 Patient Name: LYNN MYRICK

📊 Row Count Validation [Table 1]
----------------------------------------------------------------------
✅ row_count detected=3     | extracted=3     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ total_charge              computed=211.0      | extracted=211.0      match
✅ plan_allowed_amount       computed=211.0      | extracted=211.0      match
✅ deductible_amount         computed=94.0       | extracted=94.0       match
✅ copay_amount              computed=0.0        | extracted=0.0        match
✅ ineligible_amount         computed=94.0       | extracted=94.0       match
✅ previous_pay              computed=0.0        | extracted=0.0        match
✅ refund_amount             computed=0.0        | extracted=0.0      

# Last

In [2]:
import os

folder_path = r"/home/cipl/users/OCR_Project/Careington/pdfs"

for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(folder_path, filename)

        print(f"\n{'='*80}")
        print(f"Processing: {filename}")
        print(f"{'='*80}")

        try:
            run_pipeline(pdf_path)
            print(f"✅ Completed: {filename}")

        except Exception as e:
            print(f"❌ Failed: {filename}")
            print(f"Error: {e}")


Processing: Pmt_EOP_271302227.pdf
Processing Page 1
   ↳ Kept original
Processing Page 2
   ↳ Rotated 90°
Processing Page 3
   ↳ Rotated 90°

Rotated PDF saved: EOB_OUTPUT/careington_results/Pmt_EOP_271302227/Pmt_EOP_271302227_rotated.pdf
Claim Status: not denied

Scanning Page 1
Skipping page - anchors not found.

Scanning Page 2
Skipping page - anchors not found.

Scanning Page 3


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `repetition_penalty` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved : EOB_OUTPUT/careington_results/Pmt_EOP_271302227/page_3_block_1.png
📌 Expected physical service rows: 4
 Patient Name: CHARLOTTE DEBOER

📊 Row Count Validation [Table 1]
----------------------------------------------------------------------
✅ row_count detected=4     | extracted=4     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ total_charge              computed=620.0      | extracted=620.0      match
✅ plan_allowed_amount       computed=572.0      | extracted=572.0      match
✅ deductible_amount         computed=0.0        | extracted=0.0        match
✅ ineligible_amount         computed=248.2      | extracted=248.2      match
✅ previous_pay              computed=0.0        | extracted=0.0        match
✅ refund_amount             computed=0.0        | extracted=0.0        match
✅ claim_paid                computed=371.8      | extracted=371.

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved : EOB_OUTPUT/careington_results/Pmt_EOP_314231639/page_3_block_1.png
📌 Expected physical service rows: 2
 Patient Name: GARY CATTEL

📊 Row Count Validation [Table 1]
----------------------------------------------------------------------
✅ row_count detected=2     | extracted=2     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ total_charge              computed=145.0      | extracted=145.0      match
✅ plan_allowed_amount       computed=145.0      | extracted=145.0      match
✅ deductible_amount         computed=0.0        | extracted=0.0        match
✅ ineligible_amount         computed=0.0        | extracted=0.0        match
✅ previous_pay              computed=0.0        | extracted=0.0        match
✅ refund_amount             computed=0.0        | extracted=0.0        match
✅ claim_paid                computed=145.0      | extracted=145.0    

...

Saved : EOB_OUTPUT/careington_results/Pmt_EOP_538836358/page_3_block_1.png
📌 Expected physical service rows: 5
 Patient Name: LYNN MYRICK

📊 Row Count Validation [Table 1]
----------------------------------------------------------------------
✅ row_count detected=5     | extracted=5     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ total_charge              computed=210.0      | extracted=210.0      match
✅ plan_allowed_amount       computed=210.0      | extracted=210.0      match
✅ deductible_amount         computed=41.0       | extracted=41.0       match
✅ ineligible_amount         computed=41.0       | extracted=41.0       match
✅ previous_pay              computed=0.0        | extracted=0.0        match
✅ refund_amount             computed=0.0        | extracted=0.0        match
✅ claim_paid                computed=169.0      | extracted=169.0    

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


Claim Status: not denied

Scanning Page 1
Skipping page - anchors not found.

Scanning Page 2
Skipping page - anchors not found.

Scanning Page 3
Saved : EOB_OUTPUT/careington_results/Pmt_EOP_453571987/page_3_block_1.png
📌 Expected physical service rows: 3


[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Patient Name: LYNN MYRICK

📊 Row Count Validation [Table 1]
----------------------------------------------------------------------
✅ row_count detected=3     | extracted=3     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ total_charge              computed=211.0      | extracted=211.0      match
✅ plan_allowed_amount       computed=211.0      | extracted=211.0      match
✅ deductible_amount         computed=94.0       | extracted=94.0       match
✅ ineligible_amount         computed=94.0       | extracted=94.0       match
✅ previous_pay              computed=0.0        | extracted=0.0        match
✅ refund_amount             computed=0.0        | extracted=0.0        match
✅ claim_paid                computed=117.0      | extracted=117.0      match
✅ patient_responsibility    computed=94.0       | extracted=94.0       match
✅ [Table 1] Validation PAS

In [2]:
run_pipeline(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/Careington/pdfs/Pmt_EOP_267330950.pdf")

Processing Page 1
   ↳ Kept original
Processing Page 2
   ↳ Rotated 90°
Processing Page 3
   ↳ Rotated 90°

Rotated PDF saved: EOB_OUTPUT/careington_results/Pmt_EOP_267330950/Pmt_EOP_267330950_rotated.pdf
Claim Status: not denied

Scanning Page 1
Skipping page - anchors not found.

Scanning Page 2
Skipping page - anchors not found.

Scanning Page 3


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `repetition_penalty` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved : EOB_OUTPUT/careington_results/Pmt_EOP_267330950/page_3_block_1.png
 Patient Name: KEVON MAKELL

📊 Row Count Validation [Table 1]
----------------------------------------------------------------------
✅ row_count detected=2     | extracted=2     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ total_charge              computed=229.61     | extracted=229.61     match
✅ plan_allowed_amount       computed=165.0      | extracted=165.0      match
✅ deductible_amount         computed=100.0      | extracted=100.0      match
✅ copay_amount              computed=0.0        | extracted=0.0        match
✅ ineligible_amount         computed=171.21     | extracted=171.21     match
✅ previous_pay              computed=0.0        | extracted=0.0        match
✅ refund_amount             computed=0.0        | extracted=0.0        match
✅ claim_paid               

[{'eob_id': 'Pmt_EOP_267330950',
  'file_name': 'Pmt_EOP_267330950.pdf',
  'claim_status': 'not denied',
  'payor': 'CAREINGTON BENEFIT SOLUTIONS',
  'confidence_score': 100.0,
  'patients': [{'patient_name': 'KEVON MAKELL',
    'date_of_service': '07/27/2022',
    'services': [{'procedure_code': 'D0120',
      'total_charge': '$65.63',
      'plan_allowed_amount': '$43.00',
      'deductible_amount': '$0.00',
      'copay_amount': '$0.00',
      'ineligible_amount': '$22.63',
      'previous_pay': '$0.00',
      'refund_amount': '$0.00',
      'claim_paid': '$43.00',
      'patient_responsibility': '$0.00'},
     {'procedure_code': 'D4910',
      'total_charge': '$163.98',
      'plan_allowed_amount': '$122.00',
      'deductible_amount': '$100.00',
      'copay_amount': '$0.00',
      'ineligible_amount': '$148.58',
      'previous_pay': '$0.00',
      'refund_amount': '$0.00',
      'claim_paid': '$15.40',
      'patient_responsibility': '$106.60'}],
    'totals': {'total_charge': '